# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

The researchers tried to find differential gene expression in a mice model in mice with and without oxycodone withdrawal. By inhibition of these maladapted regulatory pathways they wanted to show options in managing withdrawal symptoms in mice.

What do the conditions mean?

oxy: oxycodone-treated


sal: saline-treated

What do the genotypes mean?

SNI:spared nerve injury. Mice with nerve injury and untreated chronic pain.


Sham: control group. These mice are otherwise healthy. This is an important control group to show, that the maladaption is not triggered by the condition itself.

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do?



Which groups would you compare to each other?

you treat the Sham and SNI groups separate and compare the oxy to the sal condition. Afterwards you can use the Sham to show, whether this effect occurs on ly in conjunction with a chronic pain disease or also when absent of illness.

Please also mention which outcome you would expect to see from each comparison.
A saline treatment should not induce changed behaviour in the mice, while the oxy-group should show changes in behaviour. When comparing to the Sham group later i would expect similiar results

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>
Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [2]:
import pandas as pd

df = pd.read_excel("conditions_runs_oxy_project.xlsx")
print(df.head())

  Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
0       ?  SRR23195505       x      NaN              x            NaN   
1       ?  SRR23195506       x      NaN            NaN              x   
2       ?  SRR23195507       x      NaN              x            NaN   
3       ?  SRR23195508       x      NaN            NaN              x   
4       ?  SRR23195509       x      NaN            NaN              x   

  Genotype: SNI Genotype: Sham  
0             x            NaN  
1           NaN              x  
2           NaN              x  
3             x            NaN  
4             x            NaN  


In [6]:
sal = "condition: Sal"
oxy = "Condition: Oxy"
sni = "Genotype: SNI"
sham = "Genotype: Sham"
print(df[sal].unique())
print(df[oxy].unique())
print(df[sni].unique())
print(df[sham].unique())
print()
print(f"Sal-samples: {len(df[sal].iloc[df[sal] == "x"])}")
print(f"Oxy-samples: {len(df[oxy].iloc[df[oxy] == "x"])}")
print(f"SNI-samples: {len(df[sni].iloc[df[sni] == "x"])}")
print(f"Sham-samples: {len(df[sham].iloc[df[sham] == "x"])}")

<StringArray>
['x', nan]
Length: 2, dtype: str
<StringArray>
[nan, 'x']
Length: 2, dtype: str
<StringArray>
['x', nan]
Length: 2, dtype: str
<StringArray>
[nan, 'x']
Length: 2, dtype: str
Sal-samples: 8
Oxy-samples: 8
SNI-samples: 8
Sham-samples: 8


In [8]:
sni_df = df.iloc[df[sni] == "x"].copy()
sham_df = df.iloc[df[sham] == "x"].copy()

print("SNI Samples:")
print(f"Sal-samples: {len(sni_df[sal].iloc[sni_df[sal] == "x"])}")
print(f"Oxy-samples: {len(sni_df[oxy].iloc[sni_df[oxy] == "x"])}")
print()
print("Sham Samples:")
print(f"Sal-samples: {len(sham_df[sal].iloc[sham_df[sal] == "x"])}")
print(f"Oxy-samples: {len(sham_df[oxy].iloc[sham_df[oxy] == "x"])}")
print()

SNI Samples:
Sal-samples: 4
Oxy-samples: 4

Sham Samples:
Sal-samples: 4
Oxy-samples: 4



They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [9]:
counts_df = pd.read_csv("base_counts.csv")
print(counts_df.head())

           Run       Bases
0  SRR23195505  6922564500
1  SRR23195506  7859530800
2  SRR23195507  8063298900
3  SRR23195508  6927786900
4  SRR23195509  7003550100


In [11]:
df = pd.merge(df, counts_df, on="Run", how="left")
print(df.head())

  Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
0       ?  SRR23195505       x      NaN              x            NaN   
1       ?  SRR23195506       x      NaN            NaN              x   
2       ?  SRR23195507       x      NaN              x            NaN   
3       ?  SRR23195508       x      NaN            NaN              x   
4       ?  SRR23195509       x      NaN            NaN              x   

  Genotype: SNI Genotype: Sham       Bases  
0             x            NaN  6922564500  
1           NaN              x  7859530800  
2           NaN              x  8063298900  
3             x            NaN  6927786900  
4             x            NaN  7003550100  


In [19]:
print(df.sort_values(by="Bases", ascending=True).head(2))
df.sort_values(by="Bases", ascending=True).head(2)["Run"].to_csv("samples.csv", header=False, index=False)

   Patient          Run RNA-seq  DNA-seq condition: Sal Condition: Oxy  \
11       ?  SRR23195516       x      NaN            NaN              x   
6        ?  SRR23195511       x      NaN            NaN              x   

   Genotype: SNI Genotype: Sham       Bases  
11             x            NaN  6203117700  
6            NaN              x  6456390900  


In [1]:
!nextflow run nf-core/fetchngs --input ./samples.csv --outdir ./results/Download -profile docker -c nextflow.config -resume


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/fetchngs` [drunk_monod] revision: 955c892d80 [master]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/fetchngs 1.13.0
------------------------------------------------------

Input/output options
  input              : ./samples.csv
  outdir             : ./results/Download

Deprecated options
  trace_report_suffix: 2026-09-29_16-46-36

Core Nextflow options
  revision           : master
  runName            : drunk_monod
  containerEngine    : docker
  launchDir          : /mnt/c/Users/langm/PycharmProjects/computational_workflows/computational-workflows-2026/day_02
  workDir            : /mnt/c/Use

While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.